# Business Entity Resolution: cross-encoder ensemble (Kaggle GPU)

Two multilingual transformer **cross-encoders** read both records' raw text (`name | address`) and score whether they
are the same business. They are trained on the *hard* pairs of the training S1 entities and score the *uncertain*
pairs of the holdout and the test. The scores are fused with the LightGBM pipeline locally.

**Settings:** Accelerator **GPU T4 x2** · Internet **ON** · input dataset = this kit.
**Run:** *Save Version → Save & Run All (Commit)* (about 2-2.5 h), then download `ce_outputs.zip` from the Output tab.

Every stage is checkpointed (`ce_work/<tag>/ckpt`, scoring shards). Re-running any cell resumes; after a timeout add
the previous version's output as an input and it continues from its checkpoints.

## 1. Configuration

In [ ]:
import os, sys, json, time, shutil, subprocess, glob
from pathlib import Path

SMOKE = os.environ.get("CE_SMOKE") == "1"          # local CPU test of this notebook only
for _s in (sys.stdout, sys.stderr):                  # non-Latin record text on consoles without UTF-8
    try:
        _s.reconfigure(errors="replace")
    except Exception:
        pass

def find_kit():
    cands = [Path.cwd(), Path.cwd() / "kaggle_ce"]
    if os.path.isdir("/kaggle/input"):
        cands += [p.parent for p in Path("/kaggle/input").rglob("run.py")]
    for c in cands:
        if (c / "run.py").exists() and (c / "data" / "train_pairs.parquet").exists():
            return c
    raise FileNotFoundError("kit not found: add the dataset containing run.py and data/ as a notebook input")

KIT = find_kit()
DATA = KIT / "data"
WORK = Path("/kaggle/working") if os.path.isdir("/kaggle/working") else Path.cwd() / "ce_run"
WORK.mkdir(parents=True, exist_ok=True)
OUT = WORK / "ce_work"
ANALYSIS = WORK / "analysis"
ANALYSIS.mkdir(parents=True, exist_ok=True)
RUN = WORK / "run.py"
shutil.copy(KIT / "run.py", RUN)

# a previous notebook version's output added as input -> resume from its checkpoints
RESUME_FROM = None
if os.path.isdir("/kaggle/input"):
    for p in Path("/kaggle/input").rglob("last.pt"):
        if p.parent.name == "ckpt" and p.parents[2].name == "ce_work":
            RESUME_FROM = str(p.parents[2]); break

MODELS = [
    dict(tag="e5s",  model="intfloat/multilingual-e5-small", bs=128, lr=5e-5, epochs=3.0, max_len=96),
    dict(tag="xlmr", model="FacebookAI/xlm-roberta-base",    bs=64,  lr=2e-5, epochs=2.0, max_len=96),
]
if SMOKE:
    MODELS = [dict(tag="smoke", model=os.environ.get("CE_SMOKE_MODEL", "FacebookAI/xlm-roberta-base"),
                   bs=8, lr=5e-5, epochs=1.0, max_len=48)]
print("kit:", KIT, "| work:", WORK, "| resume from:", RESUME_FROM, "| smoke:", SMOKE)
print(json.dumps(json.loads((DATA / "manifest.json").read_text()), indent=1))

## 2. Environment check

In [ ]:
import platform, socket
import torch
info = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
        "gpus": torch.cuda.device_count()}
try:
    import transformers; info["transformers"] = transformers.__version__
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(KIT / "requirements.txt")], check=True)
    import transformers; info["transformers"] = transformers.__version__
GPU_OK = []
for i in range(torch.cuda.device_count()):
    name, cap = torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i)
    try:                                  # real kernel test (P100 = sm_60 can be unsupported by recent builds)
        x = torch.randn(512, 512, device=f"cuda:{i}")
        with torch.autocast("cuda", dtype=torch.float16):
            y = (x @ x).float().mean().item()
        GPU_OK.append(i); status = "ok"
    except Exception as e:
        status = f"FAILED: {e!r}"[:200]
    print(f"GPU {i}: {name} sm_{cap[0]}{cap[1]} {torch.cuda.get_device_properties(i).total_memory / 2**30:.1f} GB -> {status}")
try:
    socket.create_connection(("huggingface.co", 443), timeout=6).close(); info["internet"] = True
except OSError:
    info["internet"] = False
info["disk_free_gb"] = round(shutil.disk_usage(WORK).free / 2**30, 1)
print(info)
if not GPU_OK and not SMOKE:
    print("WARNING: no usable GPU - training would run on CPU (very slow). Select GPU T4 x2.")
if not info["internet"] and not SMOKE:
    print("WARNING: internet off - models cannot download. Turn Internet ON or attach them as inputs.")

## 3. Data overview (analysis artifacts: `analysis/eda_*`)

In [ ]:
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

tr = pd.read_parquet(DATA / "train_pairs.parquet")
ho = pd.read_parquet(DATA / "holdout_pairs.parquet")
te = pd.read_parquet(DATA / "test_pairs.parquet")
eda = {"train_pairs": len(tr), "train_pos_rate": float(tr.y.mean()), "train_val_share": float(tr.is_val.mean()),
       "holdout_pairs": len(ho), "holdout_pos_rate": float(ho.y.mean()), "test_pairs": len(te),
       "test_by_country": te.cty.value_counts().to_dict(), "train_by_country": tr.cty.value_counts().to_dict()}
print(json.dumps(eda, indent=1))
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
bins = np.linspace(0, 1, 41)
for lab, g in tr.groupby("y"):
    ax[0].hist(g.p_gbdt, bins=bins, alpha=0.6, label=f"y={lab}", log=True)
ax[0].set_title("train pairs: LightGBM OOF p by label"); ax[0].legend()
for lab, g in ho.groupby("y"):
    ax[1].hist(g.p_gbdt, bins=bins, alpha=0.6, label=f"y={lab}", log=True)
ax[1].set_title("holdout (uncertain band): LightGBM p by label"); ax[1].legend()
ax[2].hist(te.p_gbdt, bins=bins, log=True, color="gray"); ax[2].set_title("test (uncertain band): LightGBM p")
plt.tight_layout(); plt.savefig(ANALYSIS / "eda_gbdt_p.png", dpi=110); plt.show()

rec = pd.read_parquet(DATA / "records.parquet").drop_duplicates("id").set_index("id")
sample = tr.sample(min(len(tr), 8), random_state=1)
for _, r in sample.iterrows():
    a, b = rec.loc[r.s1_id], rec.loc[r.cand_id]
    print(f"y={r.y} p_gbdt={r.p_gbdt:.3f} | {a['name']} | {a['addr']}  <->  {b['name']} | {b['addr']}")
lens = (rec["name"].fillna("").str.len() + rec["addr"].fillna("").str.len())
eda["record_chars_p50_p95_p99"] = [float(x) for x in lens.quantile([0.5, 0.95, 0.99])]
(ANALYSIS / "eda_summary.json").write_text(json.dumps(eda, indent=1, default=str))
del rec

## 4. Launch training + scoring (background, one model per GPU; safe to re-run)

In [ ]:
def launch():
    procs = {}
    pidf = WORK / "pids.json"
    old = json.loads(pidf.read_text()) if pidf.exists() else {}
    gpus = GPU_OK or [None]
    groups = {}                                    # gpu -> models (run sequentially on that GPU)
    for i, m in enumerate(MODELS):
        groups.setdefault(gpus[i % len(gpus)], []).append(m)
    for g, ms in groups.items():
        cmds = []
        for m in ms:
            if (OUT / m["tag"] / "SCORE_DONE").exists():
                print(f"{m['tag']}: already finished"); continue
            c = [sys.executable, str(RUN), "all", "--data", str(DATA), "--out", str(OUT), "--model", m["model"],
                 "--tag", m["tag"], "--bs", str(m["bs"]), "--lr", str(m["lr"]), "--epochs", str(m["epochs"]),
                 "--max-len", str(m["max_len"])]
            if g is not None: c += ["--gpu", str(g)]
            if RESUME_FROM: c += ["--resume-from", RESUME_FROM]
            if SMOKE: c += ["--smoke"]
            cmds.append(" ".join(f'"{x}"' for x in c))
        if not cmds:
            continue
        key = f"gpu{g}"
        pid = old.get(key)
        if pid and Path(f"/proc/{pid}").exists():
            print(f"{key}: still running (pid {pid})"); procs[key] = pid; continue
        logf = open(WORK / f"train_{key}.log", "a")
        p = subprocess.Popen(" && ".join(cmds), shell=True, stdout=logf, stderr=subprocess.STDOUT, cwd=str(WORK))
        procs[key] = p.pid
        print(f"{key}: started pid {p.pid} -> {[m['tag'] for m in ms]}")
    pidf.write_text(json.dumps(procs))
    return procs

PROCS = launch()

## 5. Live monitor (blocks until every model is done; interrupting it does not stop training)

In [ ]:
def status():
    rows = []
    for m in MODELS:
        pf = OUT / m["tag"] / "progress.json"
        p = json.loads(pf.read_text()) if pf.exists() else {"stage": "starting"}
        le, be = p.get("last_eval") or {}, p.get("best") or {}
        rows.append({"tag": m["tag"], "stage": p.get("stage"), "pct": p.get("pct"),
                     "eta_min": round(p["eta_min"], 1) if p.get("eta_min") is not None else None,
                     "loss": round(p["loss_ema"], 4) if p.get("loss_ema") else None,
                     "val_auc": le.get("val_auc"), "best_logloss": be.get("logloss"),
                     "pairs/s": round(p.get("pairs_per_s", 0)), "updated": p.get("updated"),
                     "error": p.get("error")})
    return pd.DataFrame(rows)

def any_alive():
    pidf = WORK / "pids.json"
    pids = json.loads(pidf.read_text()).values() if pidf.exists() else []
    return any(Path(f"/proc/{p}").exists() for p in pids) if os.path.isdir("/proc") else True

try:
    from IPython.display import clear_output, display
except ImportError:
    clear_output, display = (lambda wait=False: None), print
t0 = time.time()
try:
    while True:
        st = status()
        clear_output(wait=True); display(st); print(f"elapsed {(time.time() - t0) / 60:.1f} min")
        done = all((OUT / m["tag"] / "SCORE_DONE").exists() for m in MODELS)
        failed = st["stage"].eq("FAILED").any()
        if done or failed or (not any_alive() and time.time() - t0 > 60):
            break
        time.sleep(5 if SMOKE else 30)
except KeyboardInterrupt:
    print("monitor stopped - training continues in the background; re-run this cell to watch again")
for f in sorted(WORK.glob("train_gpu*.log")):
    print(f"--- tail of {f.name}"); print("".join(open(f).readlines()[-8:]))
if st["stage"].eq("FAILED").any():
    raise RuntimeError(f"a model failed: {st[st.stage == 'FAILED'].to_dict('records')}")

## 6. Analysis artifacts (`analysis/`): curves, CE vs LightGBM, fusion lift on the holdout, disagreements

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, roc_curve, precision_recall_curve
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression

tags = [m["tag"] for m in MODELS if (OUT / m["tag"] / "SCORE_DONE").exists()]
sig = lambda z: 1 / (1 + np.exp(-np.clip(z, -30, 30)))
glogit = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))

def mets(y, p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return {"auc": roc_auc_score(y, p), "ap": average_precision_score(y, p), "logloss": log_loss(y, p)}

# training curves
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for t in tags:
    h = pd.read_csv(OUT / t / "train_log.csv")
    ax[0].plot(h.step, h.loss_ema, label=t); ax[1].plot(h.step, h.val_auc, marker="o", label=f"{t} CE")
    if "val_gbdt_auc" in h: ax[1].axhline(h.val_gbdt_auc.iloc[-1], ls="--", color="gray")
ax[0].set_title("training loss (EMA)"); ax[1].set_title("validation AUC (dashed: LightGBM)")
ax[0].legend(); ax[1].legend(); plt.tight_layout(); plt.savefig(ANALYSIS / "training_curves.png", dpi=110); plt.show()

summary = {}
for split in ("val", "holdout"):
    df = None
    for t in tags:
        s = pd.read_parquet(OUT / t / f"scores_{split}.parquet")
        df = s if df is None else df.merge(s[["s1_id", "cand_id", f"ce_{t}"]], on=["s1_id", "cand_id"])
    if df is None:
        continue
    y = df.y.to_numpy()
    X = np.column_stack([glogit(df.p_gbdt.to_numpy())] + [df[f"ce_{t}"].to_numpy() for t in tags])
    res = {"lightgbm": mets(y, df.p_gbdt.to_numpy())}
    for k, t in enumerate(tags):
        res[f"ce_{t}"] = mets(y, sig(X[:, k + 1]))
    res["ce_mean"] = mets(y, sig(X[:, 1:].mean(1)))
    # stacking (what the local fusion does): logistic regression, out-of-fold by city folds on the holdout
    folds = df["fold"].to_numpy() if "fold" in df else (pd.util.hash_array(df.s1_id.to_numpy()) % 5)
    oof = np.zeros(len(df))
    for f in np.unique(folds):
        lr = LogisticRegression(C=1.0, max_iter=500).fit(X[folds != f], y[folds != f])
        oof[folds == f] = lr.predict_proba(X[folds == f])[:, 1]
    res["fusion_stack_oof"] = mets(y, oof)
    df["p_fused"] = oof
    summary[split] = {k: {m: round(float(v), 5) for m, v in r.items()} for k, r in res.items()}
    print(f"== {split} ({len(df):,} uncertain pairs, pos rate {y.mean():.3f})")
    print(pd.DataFrame(summary[split]).T.to_string())
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
    curves = {"lightgbm": df.p_gbdt.to_numpy(), "fusion": oof, **{f"ce_{t}": sig(df[f"ce_{t}"].to_numpy()) for t in tags}}
    for k, p in curves.items():
        fpr, tpr, _ = roc_curve(y, p); ax[0].plot(fpr, tpr, label=k)
        pr, rc, _ = precision_recall_curve(y, p); ax[1].plot(rc, pr, label=k)
        fp, mp = calibration_curve(y, p, n_bins=15, strategy="quantile"); ax[2].plot(mp, fp, marker=".", label=k)
    ax[0].set_title(f"{split}: ROC"); ax[1].set_title(f"{split}: precision-recall"); ax[2].plot([0, 1], [0, 1], "k--")
    ax[2].set_title(f"{split}: calibration"); [a.legend() for a in ax]
    plt.tight_layout(); plt.savefig(ANALYSIS / f"{split}_roc_pr_calibration.png", dpi=110); plt.show()
    if split == "holdout":
        rec = pd.read_parquet(DATA / "records.parquet").drop_duplicates("id").set_index("id")
        best_ce = max(tags, key=lambda t: summary[split][f"ce_{t}"]["auc"])
        df["ce_p"] = sig(df[f"ce_{best_ce}"].to_numpy())
        rows = []
        for title, q in (("CE says match, LightGBM says no", (df.ce_p > 0.8) & (df.p_gbdt < 0.3)),
                         ("LightGBM says match, CE says no", (df.p_gbdt > 0.8) & (df.ce_p < 0.2))):
            sub = df[q].sample(min(int(q.sum()), 15), random_state=0)
            for _, r in sub.iterrows():
                a, b = rec.loc[r.s1_id], rec.loc[r.cand_id]
                rows.append({"case": title, "y": int(r.y), "p_gbdt": round(r.p_gbdt, 3), "p_ce": round(r.ce_p, 3),
                             "p_fused": round(r.p_fused, 3), "s1": f"{a['name']} | {a['addr']}",
                             "candidate": f"{b['name']} | {b['addr']}"})
        dis = pd.DataFrame(rows)
        dis.to_csv(ANALYSIS / "holdout_disagreements.csv", index=False)
        for c, g in dis.groupby("case"):
            print(f"\n### {c}: {int(q.sum())} pairs; CE right on {(g.y == (g.p_ce > 0.5)).mean():.0%} of the sample")
            print(g.drop(columns="case").head(10).to_string(index=False))
        del rec
(ANALYSIS / "summary.json").write_text(json.dumps(summary, indent=1))

## 7. Package outputs → `ce_outputs.zip` (download from the Output tab)

In [ ]:
import zipfile
zp = WORK / "ce_outputs.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for t in tags:
        for f in ["metrics.json", "train_log.csv", "config.json", "run.log"] + [f"scores_{s}.parquet" for s in ("val", "holdout", "test")]:
            if (OUT / t / f).exists():
                z.write(OUT / t / f, f"ce_work/{t}/{f}")
    for f in ANALYSIS.glob("*"):
        z.write(f, f"analysis/{f.name}")
print(f"{zp} ({zp.stat().st_size / 2**20:.1f} MB) - models: {tags}")
print("Next: download ce_outputs.zip, put it in the project root and tell Claude to run the fusion.")